## Section 01: Traditional ML

In [ ]:
# Load Libraries
import json
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error

In [ ]:
# Load and Parse Dataset
with open('MP_dev.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
# Parse soft-labels (stored as dict) and reply text
records = []
for key, item in data.items():
    reply = item.get('text', {}).get('reply', '').strip()
    soft = item.get('soft_label', {})
    if reply and '0.0' in soft and '1.0' in soft:
        records.append({
            'text': reply,
            'soft_0': soft['0.0'],
            'soft_1': soft['1.0']
        })

df = pd.DataFrame(records)
print("Parsed samples:", len(df))

In [ ]:
# Convert text to features using TF-IDF
tfidf = TfidfVectorizer(max_features=3000)
X = tfidf.fit_transform(df['text'])  # Text features
y = df[['soft_0', 'soft_1']].values  # Soft-labels

In [ ]:
# Split dataset into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# Define regression models
models = {
    "Linear Regression": LinearRegression(),
    "Support Vector Regressor": SVR(),
    "Random Forest Regressor": RandomForestRegressor(n_estimators=100, random_state=42),
    "MLP Regressor": MLPRegressor(hidden_layer_sizes=(100,), max_iter=300, random_state=42)
}

In [ ]:
# Train each model and compute Manhattan Distance (L1 loss)
results_md = []

for name, model in models.items():
    model.fit(X_train, y_train[:, 1])  # Predicting soft label 1 (ironic)
    y_pred = model.predict(X_test)     # Predicted soft scores
    manhattan_distance = mean_absolute_error(y_test[:, 1], y_pred)
    results_md.append((name, manhattan_distance))

# Display results
results_df = pd.DataFrame(results_md, columns=["Model", "Manhattan Distance (Soft Label 1)"])
print(results_df)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

In [ ]:
# Train again on soft label 1
best_model = RandomForestRegressor(n_estimators=100, random_state=42)
best_model.fit(X_train, y_train[:, 1])
y_pred = best_model.predict(X_test)
y_true = y_test[:, 1]

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,6))
sns.scatterplot(x=y_true, y=y_pred, alpha=0.5)
plt.plot([0, 1], [0, 1], '--', color='red', label='Perfect Prediction')
plt.title("Predicted vs Actual Soft Labels (Ironic)")
plt.xlabel("Actual (Ground Truth)")
plt.ylabel("Predicted (Model Output)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(8,6))
sns.kdeplot(y_true, label="Actual", shade=True)
sns.kdeplot(y_pred, label="Predicted", shade=True)
plt.title("Distribution of Soft Labels")
plt.xlabel("Probability of Irony (Label 1)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
errors = abs(y_pred - y_true)

plt.figure(figsize=(10,4))
plt.plot(errors, marker='o', linestyle='-', color='purple', alpha=0.6)
plt.title("📉 Absolute Error per Test Sample")
plt.xlabel("Sample Index")
plt.ylabel("Absolute Error")
plt.grid(True)
plt.show()

Section 02: BERT model

In [ ]:
!pip install transformers -q

In [ ]:
# Step 2: Imports
import json
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW
from torch.nn import L1Loss  # Manhattan Distance loss


In [ ]:
# Step 3: Load and parse dataset
with open("MP_dev.json", "r", encoding="utf-8") as f:
    data = json.load(f)

records = []
for key, item in data.items():
    reply = item.get("text", {}).get("reply", "").strip()
    soft = item.get("soft_label", {})
    if reply and "0.0" in soft and "1.0" in soft:
        records.append({
            "text": reply,
            "soft_0": float(soft["0.0"]),
            "soft_1": float(soft["1.0"])
        })

df = pd.DataFrame(records)
print("Dataset size:", len(df))

In [ ]:
# Step 4: Train-test split
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["text"].tolist(),
    df[["soft_0", "soft_1"]].values.tolist(),
    test_size=0.2,
    random_state=42
)


In [ ]:
# Step 5: Tokenization & Dataset class
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

class IronyDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=max_len)
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

train_dataset = IronyDataset(train_texts, train_labels, tokenizer)
test_dataset = IronyDataset(test_texts, test_labels, tokenizer)


In [ ]:
# Step 6: DataLoaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)


In [ ]:
# Step 7: Model, optimizer, and loss
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2,
    problem_type="regression"
).to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)
loss_fn = L1Loss()  # Manhattan Distance loss


In [ ]:
# Training loop
epochs = 10
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'])
        loss = loss_fn(outputs.logits, batch['labels'])
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs} - Training Loss: {total_loss/len(train_loader):.4f}")


In [ ]:
# Evaluation (Manhattan Distance)
model.eval()
predictions, true_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'])
        predictions.extend(outputs.logits.cpu().numpy())
        true_labels.extend(batch['labels'].cpu().numpy())

# Convert to arrays
predictions = torch.tensor(predictions)
true_labels = torch.tensor(true_labels)

# Compute Manhattan Distance
manhattan_distance = torch.mean(torch.abs(predictions - true_labels)).item()
print(f"Final Manhattan Distance: {manhattan_distance:.4f}")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np


In [ ]:
# Convert tensors to numpy arrays
pred_np = predictions.numpy()
true_np = true_labels.numpy()

In [ ]:
# Plot for Label 1 (Ironic probability)
plt.figure(figsize=(8,6))
sns.scatterplot(x=true_np[:,1], y=pred_np[:,1], alpha=0.5)
plt.plot([0, 1], [0, 1], '--', color='red', label='Perfect Prediction')
plt.title("Predicted vs Actual Soft Label 1 (Ironic)")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Plot for Label 1 (Ironic probability)
plt.figure(figsize=(8,6))
sns.scatterplot(x=true_np[:,1], y=pred_np[:,1], alpha=0.5)
plt.plot([0, 1], [0, 1], '--', color='red', label='Perfect Prediction')
plt.title("Predicted vs Actual Soft Label 1 (Ironic)")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.legend()
plt.grid(True)
plt.show()


## Section 03: Hyperparameter Tuning Code

In [ ]:
import itertools
import numpy as np
import torch
from torch.utils.data import DataLoader
from torch.nn import L1Loss
from torch.optim import AdamW
from transformers import BertTokenizer, BertForSequenceClassification
from sklearn.metrics import mean_absolute_error


In [ ]:
# Train-test split
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["text"], df[["soft_0", "soft_1"]].values,
    test_size=0.2, random_state=42
)


In [ ]:
# Prepare tokenizer and dataset again
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

class IronyDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.encodings = tokenizer(list(texts), truncation=True, padding=True, max_length=max_len)
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item


In [ ]:
# Hyperparameter search space
learning_rates = [2e-5, 3e-5, 5e-5]
batch_sizes = [16, 32]
epochs_list = [2, 3]
max_lens = [64, 128]

search_space = list(itertools.product(learning_rates, batch_sizes, epochs_list, max_lens))


In [ ]:
# Store results
tuning_results = []

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for lr, batch_size, num_epochs, max_len in search_space:
    print(f"Testing: LR={lr}, Batch={batch_size}, Epochs={num_epochs}, MaxLen={max_len}")

    train_dataset = IronyDataset(X_train_text, y_train, tokenizer, max_len=max_len)
    test_dataset = IronyDataset(X_test_text, y_test, tokenizer, max_len=max_len)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size)

    model = BertForSequenceClassification.from_pretrained(
        "bert-base-uncased",
        num_labels=2,
        problem_type="regression"
    ).to(device)

    optimizer = AdamW(model.parameters(), lr=lr)
    loss_fn = L1Loss()

    # Training loop
    for epoch in range(num_epochs):
        model.train()
        for batch in train_loader:
            optimizer.zero_grad()
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
            loss = loss_fn(outputs.logits, batch["labels"])
            loss.backward()
            optimizer.step()

    # Evaluation
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for batch in test_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
            preds.extend(outputs.logits.cpu().numpy())
            labels.extend(batch["labels"].cpu().numpy())

    md = mean_absolute_error(labels, preds)
    tuning_results.append((lr, batch_size, num_epochs, max_len, md))


In [ ]:
# Sort by best Manhattan Distance
tuning_results = sorted(tuning_results, key=lambda x: x[4])
for r in tuning_results[:5]:
    print(f"LR={r[0]}, Batch={r[1]}, Epochs={r[2]}, MaxLen={r[3]} -> Manhattan Distance={r[4]:.4f}")


## Retrain Best Model

In [ ]:
# Best params from tuning
best_lr = 5e-5
best_batch = 32
best_epochs = 3
best_max_len = 128

# Prepare dataset with best max_len
best_train_dataset = IronyDataset(X_train_text, y_train, tokenizer, max_len=best_max_len)
best_test_dataset = IronyDataset(X_test_text, y_test, tokenizer, max_len=best_max_len)

best_train_loader = DataLoader(best_train_dataset, batch_size=best_batch, shuffle=True)
best_test_loader = DataLoader(best_test_dataset, batch_size=best_batch)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Model setup
best_model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2,
    problem_type="regression"
).to(device)

optimizer = AdamW(best_model.parameters(), lr=best_lr)
loss_fn = L1Loss()

# Training loop
for epoch in range(best_epochs):
    best_model.train()
    total_loss = 0
    for batch in best_train_loader:
        optimizer.zero_grad()
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = best_model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
        loss = loss_fn(outputs.logits, batch["labels"])
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{best_epochs} - Loss: {total_loss/len(best_train_loader):.4f}")


In [ ]:
best_model.eval()
preds, labels = [], []

with torch.no_grad():
    for batch in best_test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = best_model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
        preds.extend(outputs.logits.cpu().numpy())
        labels.extend(batch["labels"].cpu().numpy())

preds = np.array(preds)
labels = np.array(labels)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Scatter plot for Label 1 (Ironic)
plt.figure(figsize=(8,6))
sns.scatterplot(x=labels[:,1], y=preds[:,1], alpha=0.5)
plt.plot([0, 1], [0, 1], '--', color='red', label='Perfect Prediction')
plt.title("Improved BERT: Predicted vs Actual Soft Label 1 (Ironic)")
plt.xlabel("Actual Probability")
plt.ylabel("Predicted Probability")
plt.legend()
plt.grid(True)
plt.show()

# KDE plot (distribution comparison)
plt.figure(figsize=(8,6))
sns.kdeplot(labels[:,1], label="Actual", shade=True)
sns.kdeplot(preds[:,1], label="Predicted", shade=True)
plt.title("Improved BERT: Distribution of Soft Label 1 (Ironic)")
plt.xlabel("Probability")
plt.legend()
plt.grid(True)
plt.show()


Section 04: LLM - Mistral

In [ ]:
!pip install --upgrade transformers

In [ ]:
!pip install transformers accelerate bitsandbytes peft -q


In [ ]:
import json
import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer


In [ ]:
with open("MP_dev.json", "r", encoding="utf-8") as f:
    data = json.load(f)

records = []
for key, item in data.items():
    reply = item.get("text", {}).get("reply", "").strip()
    soft = item.get("soft_label", {})
    if reply and "0.0" in soft and "1.0" in soft:
        records.append({
            "text": reply,
            "soft_0": float(soft["0.0"]),
            "soft_1": float(soft["1.0"])
        })

df = pd.DataFrame(records)

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["text"].tolist(),
    df[["soft_0", "soft_1"]].values.tolist(),
    test_size=0.2,
    random_state=42
)


In [ ]:
import os
from huggingface_hub import login
login(os.environ["HF_TOKEN"])  # set HF_TOKEN as an environment variable, never hardcode tokens




In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", torch_dtype="auto")


In [ ]:
text = "Guy is my longtime fb friend. Very measured and rational person. Goes to show what happens to you if you criticize the PM or his riotous followers if you don't have any political backing and you live in BJP ruled state. @USER Lately Barak Valley has become a lot more intolerant"

prompt = f"""
Given the following post-reply exchange, estimate how likely it is that the reply is ironic.

Please respond only in t  his exact format:
Not Ironic: X%, Ironic: Y%

Exchange:
{text}
"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=100)
decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(decoded)


In [ ]:
import re

match = re.search(r"Not Ironic: (\d+)%.*Ironic: (\d+)%", decoded, re.DOTALL)
if match:
    not_ironic = int(match.group(1)) / 100
    ironic = int(match.group(2)) / 100
    print(f"Soft label: [Not Ironic: {not_ironic:.2f}, Ironic: {ironic:.2f}]")
else:
    print("Could not parse:", decoded)


In [ ]:
records = []
for k, v in data.items():
    post = v['text']['post']
    reply = v['text']['reply']
    soft = v['soft_label']
    records.append({
        'id': k,
        'text': post + " " + reply,
        'soft_label': [soft.get("0.0", 0.0), soft.get("1.0", 0.0)]
    })

df = pd.DataFrame(records)


In [ ]:
sample_df = df.sample(100, random_state=42).reset_index(drop=True)
texts = sample_df['text'].tolist()
true_soft_labels = sample_df['soft_label'].tolist()


In [ ]:
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "mistralai/Mistral-7B-Instruct-v0.2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto").to("cuda") # Move model to CUDA

pred_soft_labels = []

# Prepare prompts for batch
prompts = [
    f"""
Given the following post-reply exchange, estimate how likely it is that the reply is ironic.

Please respond **only** in this exact format:
Not Ironic: X%, Ironic: Y%

Exchange:
{text}
""" for text in texts
]

# Tokenize all prompts together (batch)
inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True).to("cuda")

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=20,  # reduce number of generated tokens
        do_sample=False,    # greedy decoding for speed
        pad_token_id=tokenizer.eos_token_id  # avoid warning for no pad token
    )

# Decode output for each input in batch
decoded_outputs = tokenizer.batch_decode(outputs, skip_special_tokens=True)

for decoded in decoded_outputs:
    match = re.search(r"Not Ironic: (\d+)%.*Ironic: (\d+)%", decoded, re.DOTALL)
    if match:
        try:
            not_ironic = int(match.group(1)) / 100
            ironic = int(match.group(2)) / 100
            pred_soft_labels.append([not_ironic, ironic])
        except ValueError:
            print(f"Could not parse percentages from: {decoded}")
            pred_soft_labels.append([0.5, 0.5])  # fallback
    else:
        print("Could not parse:", decoded)
        pred_soft_labels.append([0.5, 0.5])  # fallback

# print(pred_soft_labels)

In [ ]:
true_soft_labels = np.array(true_soft_labels)
pred_soft_labels = np.array(pred_soft_labels)

manhattan_distances = np.abs(true_soft_labels - pred_soft_labels).mean(axis=1)
mean_manhattan = manhattan_distances.mean()

print(f"\n Mean Manhattan Distance on 100 samples: {mean_manhattan:.4f}")


Another LLM

In [ ]:
# Install dependencies
!pip install transformers datasets accelerate peft bitsandbytes -q


In [ ]:
# Imports
import json
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
from sklearn.metrics import mean_absolute_error
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer


In [ ]:
# Load and prepare dataset
with open("MP_dev.json", "r", encoding="utf-8") as f:
    data = json.load(f)

records = []
for key, item in data.items():
    reply = item.get("text", {}).get("reply", "").strip()
    post = item.get("text", {}).get("post", "").strip()
    soft = item.get("soft_label", {})
    if reply and '0.0' in soft and '1.0' in soft:
        combined_text = f"Post: {post}\nReply: {reply}"
        records.append({
            "text": combined_text,
            "soft_0": float(soft['0.0']),
            "soft_1": float(soft['1.0'])
        })

df = pd.DataFrame(records)

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["text"].tolist(),
    df[["soft_0", "soft_1"]].values.tolist(),
    test_size=0.2,
    random_state=42
)


In [ ]:
# Tokenizer & Dataset class
model_name = "distilbert/distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

class SoftLabelDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=max_len)
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

train_dataset = SoftLabelDataset(train_texts, train_labels, tokenizer)
test_dataset = SoftLabelDataset(test_texts, test_labels, tokenizer)

In [ ]:
# Load model for regression
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    problem_type="regression",
    torch_dtype=torch.float32,
    device_map="auto"
)
model.config.pad_token_id = tokenizer.pad_token_id


In [ ]:
# Manhattan Distance metric
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    md = mean_absolute_error(labels, preds)
    return {"manhattan_distance": md}


In [ ]:
# Training arguments (FP16/BF16 disabled)
training_args = TrainingArguments(
    output_dir="./llm_results",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,
    eval_strategy="epoch",
    logging_dir="./logs",
    save_strategy="no",
    fp16=False,
    bf16=False
)


In [ ]:
# Trainer setup
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)


In [ ]:
trainer.train()


In [ ]:
# Evaluate
metrics = trainer.evaluate()
print("Final Manhattan Distance:", metrics["eval_manhattan_distance"])
